#1~4で最も高いスコアを記録したデータ・モデルを用いて、さらに改良を行う。

In [2]:
#import
import pandas as pd
from sklearn.model_selection import train_test_split  #データの分割

from sklearn.preprocessing import StandardScaler #標準化
from sklearn.model_selection import KFold #K分割交差検証
from sklearn.model_selection import cross_validate  #K分割交差検証

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰

from statsmodels.stats.outliers_influence import variance_inflation_factor  #VIF
from sklearn.decomposition import PCA  #PCA

In [3]:
#データフレームの読み込み
'''
以降はスコアが最高であったdf3を用いて分析を行う。
sc_x,df_yはdf3を基に作成する。
ただし、必要があればdf1, df2も利用する。
'''
df3 = pd.read_csv('datafiles/df3_lowered_VIF.csv')

sc_x = df3.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df3['SalePrice'])

In [4]:
#K分割交差検証のための準備
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [5]:
#リッジ回帰のalphaを90~1000で最適化（以前は１～100までの自然数で実験しただけであった）
best_ridgescore = 0
best_alpha = 0

for i in range(90,1000):
    ridgeModel = Ridge(random_state = 0, alpha = i)
    all_result = cross_validate(ridgeModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_ridgescore:
        best_ridgescore = result
        best_alpha = i
print(f'最適な正則化項＝{best_alpha}　最高スコア＝{best_ridgescore}')

最適な正則化項＝478　最高スコア＝0.8205844141029276


In [6]:
#リッジ回帰のalphaを478付近でより細かく最適化（以前は１～100までの自然数で実験しただけであった）
best_ridgescore = 0
best_alpha = 0

for i in range(47700,47900):
    num = i/100
    ridgeModel = Ridge(random_state = 0, alpha = num)
    all_result = cross_validate(ridgeModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_ridgescore:
        best_ridgescore = result
        best_alpha = num
print(f'最適な正則化項＝{best_alpha}　最高スコア＝{best_ridgescore}')


最適な正則化項＝478.14　最高スコア＝0.8205844152693619


In [7]:
#4_analysis_2においてVIFを行の削除により下げたスコアの最高値   0.7971109145093968よりもスコアが向上した。

In [8]:
#引き続き、特徴量の最適化を行う。
model5 = Ridge(random_state = 0, alpha = best_alpha)
model5.fit(sc_x, df_y)
coef_df = pd.DataFrame({
    'features':sc_x.columns,
    'Coefficient': model5.coef_
})
coef_df.sort_values('Coefficient', ascending=False)

,features,Coefficient
2,OverallQual,10957.831681
8,1stFlrSF,9102.571118
16,TotRmsAbvGrd,7851.661035
195,Neighborhood_NoRidge,6941.700492
196,Neighborhood_NridgHt,6872.611172
...,...,...
75,BldgType_TwnhsE,-3760.381477
127,KitchenQual_Gd,-4433.561103
48,BsmtQual_Gd,-4760.329032
97,PoolQC_Gd,-4799.248606


In [36]:
'''
今回のデータセットでは各特徴量の意味が明確であるから、ドメイン知識に基づいて仮説検証を行う。
以下が検証すべき仮説である。

・都会の一戸建ては価格が高い。
・好立地で敷地面積が大きいと価格は高くなりやすい。
・都会の中でも住宅街のような閑静なエリアは価格が高い。
・寒い地域で暖房設備に欠陥があったり、断熱性の弱い外壁素材であったりすると価格が低い。
・プール、地下室など生活のために必須ではない要素がある家には富裕層が住んでいる可能性が高いため価格も高い。
・特に立地・住宅の種類は、他の様々な特徴量と関連性が強そうである。

ゆえに、以下の順番で検証を行う。
1.Neighborhood（立地）に関連する各ダミー変数列と、各特徴量の交互作用特徴量を作成
2.MSSubClass（住宅の種類）と、各特徴量の交互作用特徴量を作成
3.BsmtQual_NA（地下室の有無）、WoodDeckSF、OpenPorchSF、EnclosedPorch、3SsnPorch、ScreenPorch、
  PoolArea、MiscFeature（家に付随する必須ではない機能）は関連性が高い可能性があるので、
  PCAにより特徴量を統合することや、クラスタリングでの分類を目指す。

検証のために必要な特徴量を削除してしまっていたので、以下ではdf1（特徴量削除前のデータフレーム）を用いる。
'''
df1 = pd.read_csv('datafiles/df1_all_col.csv')

sc_x = df1.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df3['SalePrice'])


In [37]:
result = cross_validate(model5, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel2のスコア＝0.8192748628393769


In [38]:
#元の列をリスト化しておく
original_cols = list(sc_x.columns)

In [39]:
#1の仮説検証
#sc_x1を１の仮説検証のためのデータフレームとする。
sc_x1 = sc_x.copy()

Neighborhood_cols = []
for c in original_cols:
    if 'Neighborhood' in c:
        Neighborhood_cols.append(c)

not_neighborhood_cols = list(set(original_cols) - set(Neighborhood_cols))

#Neighborhood_colsそれぞれに対して多項式特徴量を追加
new_cols = {}
for c1 in Neighborhood_cols:
    for c2 in not_neighborhood_cols:
        col_name = c1 + '_' + c2
        new_cols[col_name] = sc_x1[c1] * sc_x1[c2]
sc_x1 = pd.concat([sc_x1, pd.DataFrame(new_cols, index=sc_x1.index)], axis=1)

In [40]:
#作成したデータで学習
model5 = Ridge(alpha = best_alpha)
result = cross_validate(model5, sc_x1, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel5のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel5のスコア＝0.838013880006198


In [ ]:
#1により、スコアは誤差程度向上した。ただし、余計な特徴量を非常に多く追加してしまった可能性がある。
#ゆえに、以下では特徴量のうち立地と特に強く関連しそうなものをドメイン知識により選定し、交互作用特徴量を追加する。

#sc_x1-2を１の仮説検証のためのデータフレームとする。
sc_x1_2 = sc_x.copy()

Index(['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond',
       'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
       ...
       'Condition2_Feedr', 'Condition2_Norm', 'Condition2_PosA',
       'Condition2_PosN', 'Condition2_RRAe', 'Condition2_RRAn',
       'Condition2_RRNn', 'GarageFinish_NA', 'GarageFinish_RFn',
       'GarageFinish_Unf'],
      dtype='object', length=252)


In [ ]:
'''

#立地と特に強く関連しそうな特徴量の選定
to_add_cols = ['MasVnrArea', 'Street_Pave', 'BsmtFinType1_LwQ', 'SaleType_WD', 'HouseStyle_1.5Unf', 
               'GarageCond_Po', 'HeatingQC_Gd', 'Fireplaces', 'Exterior1st_ImStucc', 'Condition1_Feedr', 
               'BldgType_2fmCon', 'OverallCond', 'BldgType_Duplex', 'Exterior1st_BrkComm', 'Exterior2nd_AsphShn', 
               'MiscFeature_TenC', 'BsmtFinType2_Rec', 'Foundation_CBlock', 'Exterior2nd_ImStucc', 'Electrical_Mix', 
               'Condition1_RRNe', 'Heating_Grav', 'Exterior2nd_Plywood', 'LotFrontage', 'GarageQual_Gd', 'RoofMatl_Metal', 
               'GarageFinish_Unf', 'Exterior2nd_Wd Sdng', 'GarageYrBlt', 'OpenPorchSF', 'RoofMatl_Roll', 
               'Exterior1st_CBlock', 'LandContour_HLS', 'SaleCondition_Partial', 'ExterCond_Po', 'BsmtFinType2_GLQ', 
               'SaleCondition_Normal', 'Foundation_PConc', 'GarageType_Basment', 'MasVnrType_Stone', 'BsmtExposure_No', 
               'Exterior1st_Stucco', 'PavedDrive_P', 'MiscVal', 'ExterQual_Fa', 'LotConfig_Inside', 'BsmtFinType1_Unf', 
               'Exterior2nd_Stone', 'LotShape_Reg', 'TotRmsAbvGrd', 'MSZoning_RM', 'Condition1_PosA', 'Condition1_RRNn', 
               'MiscFeature_Shed', 'Exterior1st_Wd Sdng', 'Electrical_NA', 'Exterior1st_AsphShn', 'BsmtExposure_Gd', 
               'BldgType_TwnhsE', 'GarageQual_Fa', 'BsmtQual_NA', 'Condition2_Feedr', 'ExterCond_Gd', 'Exterior1st_Plywood', 
               'Functional_Sev', 'WoodDeckSF', 'BsmtFinType2_LwQ', 'BldgType_Twnhs', 'LotConfig_CulDSac', 'RoofStyle_Gambrel', 
               'ExterQual_Gd', 'SaleType_Con', 'GarageQual_Po', 'PavedDrive_Y', 'BsmtCond_Gd', 'BedroomAbvGr', 
               'Condition1_RRAn', 'Condition2_PosA', 'FireplaceQu_Po', 'HeatingQC_Po', 'Exterior1st_BrkFace', 
               'Exterior2nd_BrkFace', 'BsmtQual_Fa', 'Functional_Mod', 'GarageFinish_RFn', 'GarageType_BuiltIn', 
               'GarageType_Detchd', 'Functional_Typ', 'RoofMatl_WdShngl', 'BsmtHalfBath', 'BsmtUnfSF', 'LotShape_IR2', 
               'RoofStyle_Mansard', 'Heating_Wall', 'BsmtFinType2_BLQ', 'KitchenQual_Fa', 'HouseStyle_SFoyer', 'Heating_GasW', 
               'BsmtExposure_Mn', 'HouseStyle_2.5Fin', 'SaleCondition_Family', 'Condition2_RRNn', 'Fence_MnWw', 'SaleType_ConLD', 
               'BsmtFinType1_BLQ', 'YearRemodAdd', 'GarageType_CarPort', 'OverallQual', 'LandSlope_Mod', 
               'Condition1_Norm', 'HeatingQC_TA', 'KitchenQual_TA', 'BsmtCond_Po', 'Exterior2nd_Other', 
               'Exterior2nd_Wd Shng', 'YearBuilt', 'CentralAir_Y', 'LotConfig_FR2', 'Exterior2nd_Stucco', 
               'SaleType_ConLw', 'HouseStyle_SLvl', 'YrSold', '3SsnPorch', 'Condition2_RRAe', 'RoofStyle_Hip', 
               'LandSlope_Sev', 'Fence_MnPrv', 'Foundation_Stone', 'LandContour_Lvl', 'Fence_NA', 'PoolQC_Gd', 
               'Alley_NA', 'LotArea', 'HalfBath', 'Foundation_Slab', 'Electrical_SBrkr', 'Condition2_Norm', 
               'Exterior2nd_CmentBd', 'GarageArea', 'KitchenQual_Gd', 'Exterior2nd_Brk Cmn', 'MiscFeature_Othr', 
               'Heating_OthW', 'Condition2_RRAn', 'ScreenPorch', 'FireplaceQu_Gd', 'Exterior1st_HdBoard', 'LowQualFinSF', 
               'FireplaceQu_TA', 'ExterCond_Fa', 'HouseStyle_2Story', 'Condition2_PosN', 'RoofMatl_WdShake', 'LotShape_IR3', 
               'FireplaceQu_Fa', 'Alley_Pave', 'Functional_Maj2', 'SaleCondition_Alloca', 'GarageCond_Gd', 'MSZoning_FV', 
               'Electrical_FuseP', 'HouseStyle_2.5Unf', 'GarageCars', 'RoofMatl_Membran', 'SaleType_CWD', 'EnclosedPorch', 
               'LotConfig_FR3', 'Functional_Min2', 'BsmtFinType2_Unf', 'Exterior2nd_MetalSd', 'FullBath', 'PoolQC_Fa', 
               'SaleType_Oth', 'HeatingQC_Fa', '1stFlrSF', 'BsmtFullBath', 'RoofStyle_Shed', 'Exterior1st_WdShing', 
               'Condition1_PosN', 'BsmtCond_TA', 'Electrical_FuseF', 'Exterior2nd_HdBoard', 'RoofMatl_Tar&Grv', 
               'BsmtQual_Gd', 'Exterior1st_Stone', 'LandContour_Low', 'MSZoning_RH', 'SaleType_ConLI', 'KitchenAbvGr', 
               'Functional_Min1', 'SaleCondition_AdjLand', 'HouseStyle_1Story', 'Condition1_RRAe', 'MoSold', 
                 'BsmtFinType1_GLQ', 'BsmtFinType1_Rec', 'BsmtQual_TA', 'Foundation_Wood', 'PoolArea', 'Utilities_NoSeWa', 'Fence_GdWo', 'GarageCond_Fa']
#to_add_colsそれぞれに対して多項式特徴量を追加
new_cols = {}
for c1 in Neighborhood_cols:
    for c2 in to_add_cols:
        col_name = c1 + '_' + c2
        new_cols[col_name] = sc_x1_2[c1] * sc_x1_2[c2]
sc_x1_2 = pd.concat([sc_x1_2, pd.DataFrame(new_cols, index=sc_x1_2.index)], axis=1)

#作成したデータで学習
model5 = Ridge(alpha = best_alpha)
result = cross_validate(model5, sc_x1_2, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel5のスコア＝{sum(result['test_score'])/len(result['test_score'])}')


'''